# Fine-tune Gemma for Source Code Summarization

This notebook demonstrates how to fine-tune Google's Gemma model using Unsloth (for fast Q-LoRA) on a custom dataset of code-summary pairs.

## Prerequisites
- GPU Runtime (T4 is sufficient for Gemma-2b, A100/L4 for 7b)
- `code_summary_dataset.jsonl` uploaded to the runtime or Drive

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install evaluate rouge_score bert_score

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import evaluate
import numpy as np

## 1. Load Model and Tokenizer

In [ ]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2b-it-nb", # "unsloth/gemma-7b-it" for 7b
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 2. Configure LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

## 3. Load and Format Data

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    instructions = ["Summarize the following function code, incorporating structural analysis details if available."] * len(examples['code'])
    inputs       = examples["code"]
    outputs      = examples["summary"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # Must add EOS_TOKEN, otherwise your generation will go on forever!
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# Load dataset
# Assuming 'code_summary_dataset.jsonl' is in the current directory
dataset = load_dataset("json", data_files="code_summary_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

## 4. Train the Model

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase this for real training!
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 5. Inference and Evaluation

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

inputs = tokenizer(
[
    alpaca_prompt.format(
        "Summarize the following function code.", # Instruction
        "def add(a, b):\n    return a + b", # Input
        "", # Output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

In [ ]:
# Evaluation Metrics
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

def compute_metrics(predictions, references):
    b = bleu.compute(predictions=predictions, references=references)
    r = rouge.compute(predictions=predictions, references=references)
    m = meteor.compute(predictions=predictions, references=references)
    bs = bertscore.compute(predictions=predictions, references=references, lang="en")
    
    return {
        "BLEU": b['bleu'],
        "ROUGE-L": r['rougeL'],
        "METEOR": m['meteor'],
        "BERTScore": np.mean(bs['f1'])
    }

# Example evaluation loop would go here
print("Metrics loaded. Run evaluation loop on test set.")